# Text Characteristics (Figure D2)

Replaces the draft's Figures 3, 4, 5, 7, 8 and 9 - six figures over six pages - with one
six-panel figure.

The consolidation is not only about space. The draft puts PathOPEN's word counts
(Figures 3-5) and PathVQA's (Figures 7-9) in different figures on different pages with
different axes, so the reader has to hold two scales in mind to compare them. Here every
distribution panel shares one x-axis, and panel d overlays PathVQA raw against filtered,
which shows the thing worth knowing: filtering barely moves the distribution.

Panel f plots the question-opening words, which the analysis notebooks have always
produced (`*_first_word_distribution.png`) but which appear nowhere in the draft. For a
data descriptor this is reuse-relevant - it says what kind of questions the dataset
actually contains.


In [ ]:
import os
import sys
import pickle
import collections

sys.path.insert(0, os.path.abspath("."))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import nature_style as ns

ns.apply_style()
# Panel tick labels at the shared 7 pt default read small once the figure is scaled to
# 183 mm across seven panels; 7.5 keeps them legible without breaching the type budget.
plt.rcParams.update({"xtick.labelsize": 7.5, "ytick.labelsize": 7.5,
                     "axes.labelsize": 8})

# The raw sheet, not processed/data.json: this reproduces the published numbers exactly.
# See the note below on .unique() - the counts differ slightly from the processed records
# and the paper quotes these.
RAW = os.path.join("..", "pathopen_data", "raw", "data.csv")
PATHVQA_FILTERED = os.path.join("..", "pathvqa_data_filtered", "qas")

raw = pd.read_csv(RAW)


def words(text):
    return len(str(text).split())


# Thresholds are ">= 5" and ">= 10", matching the paper and
# data_analysis/raw_pathopen_data_analysis.ipynb. Using strict ">" moves every figure
# by 2-5 points (open-ended questions 86.4% -> 82.8%), so the operator matters.
def pct_at_least(values, threshold):
    return 100.0 * sum(1 for value in values if value >= threshold) / len(values)


# Open-ended spans two question slots per case and keeps every row.
oe_question = (list(raw["Open Ended - Question 1"].dropna())
               + list(raw["Open Ended - Question 2"].dropna()))
oe_correct = (list(raw["Open Ended - Answer 1"].dropna())
              + list(raw["Open Ended - Answer 2"].dropna()))
oe_wrong = (list(raw["Open Ended - Wrong Answer 1"].dropna())
            + list(raw["Open Ended - Wrong Answer 2"].dropna()))

# MCQ and close-ended are de-duplicated per column with .unique(), as the original
# analysis does, because that is what produced the published percentages.
#
# NOTE: this drops genuinely distinct items that happen to share wording - two cases
# both asking "What is the diagnosis and based on what features?" are two real questions
# about two different images. The effect is small (157 -> 155 MCQ questions, 157 -> 154
# close-ended, 785 -> 775 options) but it is an undercount, not a correction. Kept here
# for consistency with the manuscript; flagged so the choice is visible.
mcq_question = list(raw["MCQ - Question"].dropna().unique())
mcq_option = []
for slot in range(1, 6):
    mcq_option += list(raw[f"MCQ - Option {slot}"].dropna().unique())
ce_question = list(raw["Close-Ended Question 1"].dropna().unique())

pathopen = {
    "OE question": [words(s) for s in oe_question],
    "OE correct answer": [words(s) for s in oe_correct],
    "OE wrong answer": [words(s) for s in oe_wrong],
    "MCQ question": [words(s) for s in mcq_question],
    "MCQ option": [words(s) for s in mcq_option],
    "CE question": [words(s) for s in ce_question],
}
for key, values in pathopen.items():
    print(f"{key:20s} n={len(values):5d}  mean={np.mean(values):5.1f}  "
          f">=5w {pct_at_least(values, 5):5.1f}%  >=10w {pct_at_least(values, 10):5.1f}%")

# First words, for panel f.
oe_first_words = [str(s).split()[0].lower().strip("?,") for s in oe_question]
mcq_first_words = [str(s).split()[0].lower().strip("?,") for s in mcq_question]
ce_first_words = [str(s).split()[0].lower().strip("?,") for s in ce_question]


In [ ]:
# --- PathVQA text ----------------------------------------------------------------
# "filtered" here is the H&E + question-type split (8,992 pairs), BEFORE the word-count
# floor. Plotting the post-floor set would be circular: a set selected for having >= 5
# words trivially has >= 5 words.
filtered = []
for split in ("train", "val", "test"):
    path = os.path.join(PATHVQA_FILTERED, split, f"{split}_qa.pkl")
    with open(path, "rb") as handle:
        filtered.extend(pickle.load(handle))
print(f"PathVQA filtered pairs: {len(filtered)}")

YES_NO = {"yes", "no"}
pathvqa = collections.defaultdict(list)
for row in filtered:
    answer = str(row["answer"]).strip().lower()
    bucket = "CE question" if answer in YES_NO else "OE question"
    pathvqa[bucket].append(words(row["question"]))
    if answer not in YES_NO:
        pathvqa["OE answer"].append(words(row["answer"]))

for key, values in pathvqa.items():
    print(f"{key:20s} n={len(values):5d}  mean={np.mean(values):5.1f}")

# --- raw (unfiltered) PathVQA, from the cached histogram ------------------------------
# The full corpus lives outside the repo, so its word counts are cached as a 2 KB
# histogram (see the header of the CSV). Expanding it back gives the same distributions
# and thresholds as counting the 32,795 pairs directly.
#
# This is here to close an obvious reviewer question: did H&E filtering select the
# short-answer tail? It did not - open-ended answers average 2.6 words in the raw
# corpus and 2.6 after filtering. The comparison with PathOPEN is a property of
# PathVQA, not an artefact of the subset.
RAW_CACHE = "pathvqa_raw_word_counts.csv"
raw_hist = pd.read_csv(RAW_CACHE, comment="#")
pathvqa_raw = {
    series: np.repeat(group["words"].values, group["count"].values).tolist()
    for series, group in raw_hist.groupby("series")
}
for key, values in sorted(pathvqa_raw.items()):
    print(f"raw {key:14s} n={len(values):6d}  mean={np.mean(values):5.1f}  "
          f">=5w {pct_at_least(values, 5):5.1f}%  >=10w {pct_at_least(values, 10):5.1f}%")

THRESHOLD_ROWS = [
    ("PathOPEN OE question", pathopen["OE question"]),
    ("PathOPEN OE correct answer", pathopen["OE correct answer"]),
    ("PathOPEN OE wrong answer", pathopen["OE wrong answer"]),
    ("PathOPEN MCQ question", pathopen["MCQ question"]),
    ("PathOPEN MCQ option", pathopen["MCQ option"]),
    ("PathOPEN CE question", pathopen["CE question"]),
]

# PathVQA goes in its own panel as stacked composition rather than six more threshold
# rows: raw-vs-filtered is a comparison, and twelve horizontal bars crowded out the
# distribution panels above. Three bands per bar carry the whole distribution, and the
# raw/filtered pair sits adjacent so the (non-)difference is read directly.
BANDS = [("< 5 words", 0, 5), ("5-9 words", 5, 10), ("$\\geq$ 10 words", 10, 10**6)]
PATHVQA_SERIES = ["OE question", "OE answer", "CE question"]


def composition(values):
    total = len(values)
    return [100.0 * sum(1 for v in values if low <= v < high) / total
            for _, low, high in BANDS]


print()
for series in PATHVQA_SERIES:
    for name, source in (("raw", pathvqa_raw), ("H&E", pathvqa)):
        shares = composition(source[series])
        print(f"  {name:4s} {series:12s} "
              + "  ".join(f"{label} {share:5.1f}%"
                          for (label, _, _), share in zip(BANDS, shares)))
print()
for label, values in THRESHOLD_ROWS:
    print(f"{label:24s} >=5w {pct_at_least(values, 5):5.1f}%   "
          f">=10w {pct_at_least(values, 10):5.1f}%")


In [ ]:
import matplotlib.transforms as mtransforms

PO = ns.DATASET_COLORS["PathOPEN"]
PV = "#7F7F7F"
BINS = np.arange(0, 56, 2)


def density(ax, series, xlabel, xlim=(0, 55)):
    """Overlaid step histograms, normalised to percentage of their own series.

    Counts differ by two orders of magnitude between the datasets (309 PathOPEN
    questions against 8,992 PathVQA pairs), so raw frequency would make one invisible.
    """
    for label, values, colour, style in series:
        weights = np.ones(len(values)) * 100.0 / len(values)
        ax.hist(values, bins=BINS, weights=weights, histtype="step", lw=1.1,
                color=colour, ls=style, label=f"{label} ({np.mean(values):.0f} w)")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("% of items")
    ax.set_xlim(*xlim)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.legend(fontsize=ns.fs(7), frameon=False, handletextpad=0.5,
              labelspacing=0.32, borderaxespad=0.2)


fig = plt.figure(figsize=ns.mm(ns.DOUBLE_COL_MM * 1.065, 244))
# Tight hspace: the distribution panels were being squeezed to buy space that the gaps
# were wasting. Row 3 is the tall one because panel e carries nine rows, and panel f
# shares its height rather than being pinned to the shorter upper rows.
outer = fig.add_gridspec(3, 1, height_ratios=[1.0, 1.0, 1.26], hspace=0.62)

row1 = outer[0].subgridspec(1, 2, width_ratios=[1, 1], wspace=0.26)
row2 = outer[1].subgridspec(1, 3, width_ratios=[1, 1, 1.05], wspace=0.36)
# e and f share the bottom row. e needs the width for nine long labels; f is a six-bar
# chart that reads fine narrower, and it inherits the full row height.
row3 = outer[2].subgridspec(1, 2, width_ratios=[1.95, 1], wspace=0.26)

ax = fig.add_subplot(row1[0, 0])
density(ax, [("question", pathopen["OE question"], PO, "-"),
             ("correct answer", pathopen["OE correct answer"], ns.OKABE_ITO["blue"], "-"),
             ("wrong answer", pathopen["OE wrong answer"], ns.OKABE_ITO["green"], "--")],
        "Words")
ax.set_title("PathOPEN open-ended", fontsize=7, pad=3)
ns.panel_label_below(ax, "a")

ax = fig.add_subplot(row1[0, 1])
density(ax, [("question", pathopen["MCQ question"], PO, "-"),
             ("option", pathopen["MCQ option"], ns.OKABE_ITO["blue"], "-")],
        "Words", xlim=(0, 40))
ax.set_title("PathOPEN MCQ", fontsize=7, pad=3)
ns.panel_label_below(ax, "b")

ax = fig.add_subplot(row2[0, 0])
density(ax, [("question", pathopen["CE question"], PO, "-")], "Words", xlim=(0, 30))
ax.set_title("PathOPEN close-ended", fontsize=7, pad=3)
ns.panel_label_below(ax, "c")

ax = fig.add_subplot(row2[0, 1])
# Raw dotted, H&E-filtered solid, one hue per series. The curves nearly coincide, and
# that is the statement: filtering did not select the short-answer tail.
density(ax, [("OE question", pathvqa["OE question"], PV, "-"),
             ("OE answer", pathvqa["OE answer"], ns.OKABE_ITO["vermillion"], "-")],
        "Words")          # same xlim as panel a - the comparison is the point
for values, colour in ((pathvqa_raw["OE question"], PV),
                       (pathvqa_raw["OE answer"], ns.OKABE_ITO["vermillion"])):
    weights = np.ones(len(values)) * 100.0 / len(values)
    ax.hist(values, bins=BINS, weights=weights, histtype="step", lw=0.9,
            color=colour, ls=":", zorder=1)
ax.legend(handles=[plt.Line2D([], [], color=PV, lw=1.1,
                              label=f"OE question ({np.mean(pathvqa['OE question']):.0f} w)"),
                   plt.Line2D([], [], color=ns.OKABE_ITO["vermillion"], lw=1.1,
                              label=f"OE answer ({np.mean(pathvqa['OE answer']):.0f} w)"),
                   plt.Line2D([], [], color="#666666", lw=0.9, ls=":",
                              label="raw (unfiltered)")],
          fontsize=ns.fs(7), frameon=False, handletextpad=0.5, labelspacing=0.32,
          borderaxespad=0.2)
ax.set_title("PathVQA word counts", fontsize=7, pad=3)
ns.panel_label_below(ax, "d")

# Panel e: the four threshold figures of the draft, as one chart.
ax = fig.add_subplot(row3[0, 0])
labels = [label for label, _ in THRESHOLD_ROWS]
y = np.arange(len(labels))
over5 = [pct_at_least(values, 5) for _, values in THRESHOLD_ROWS]
over10 = [pct_at_least(values, 10) for _, values in THRESHOLD_ROWS]
colours = [PO] * len(labels)
ax.barh(y - 0.23, over5, 0.38, color=colours, label="$\\geq$ 5 words")
ax.barh(y + 0.23, over10, 0.38, color=colours, alpha=0.5, label="$\\geq$ 10 words")
ax.set_yticks(y)
ax.set_yticklabels([label.replace("PathOPEN ", "")
                    .replace("correct answer", "correct ans.")
                    .replace("wrong answer", "wrong ans.")
                    .replace("OE answer", "OE ans.")
                    for label in labels], fontsize=ns.fs(6.8))
ax.invert_yaxis()
ax.set_xlabel("% of items at or above threshold")
ax.set_title("PathOPEN word-count thresholds", fontsize=7, pad=4)
ax.set_xlim(0, 108)
for index, (five, ten) in enumerate(zip(over5, over10)):
    # Both numbers always, one per bar. An earlier version suppressed the second when
    # the two were within a few points - which hid the value precisely where the two
    # thresholds agree, e.g. correct answers at 99.0 and 98.4.
    ax.text(five + 1.2, index - 0.23, f"{five:.0f}", va="center", fontsize=ns.fs(6.5),
            color="#444444")
    ax.text(ten + 1.2, index + 0.23, f"{ten:.0f}", va="center", fontsize=ns.fs(6.5),
            color="#444444")
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.legend(fontsize=ns.fs(7), frameon=False, loc="center left", ncol=1,
          handletextpad=0.5, labelspacing=0.4, borderaxespad=0.0,
          bbox_to_anchor=(0.69, 0.255))
ns.panel_label_below(ax, "f")

# Panel f: what the questions actually ask - never plotted in the draft.
ax = fig.add_subplot(row3[0, 1])
TOP = 5
counts = collections.Counter(oe_first_words)
common = [word for word, _ in counts.most_common(TOP)]
tail_pct = 100.0 * sum(count for word, count in counts.items()
                       if word not in common) / len(oe_first_words)
mcq_counts = collections.Counter(mcq_first_words)
ce_counts = collections.Counter(ce_first_words)
x = np.arange(len(common))
oe_pct = [100.0 * counts[w] / len(oe_first_words) for w in common]
ax.bar(x, oe_pct, 0.62, color=PO)
ax.bar([len(common)], [tail_pct], 0.62, color="#BFBFBF")
x = np.arange(len(common) + 1)
ax.set_xticks(x)
ax.set_xticklabels(common + [f"other\n({len(counts) - TOP})"], fontsize=ns.fs(7),
                   rotation=30, ha="right", rotation_mode="anchor")
ax.set_ylabel("% of open-ended questions")
ax.set_title("Question opening word", fontsize=7, pad=3)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
top_ce = ce_counts.most_common(3)
ax.set_ylim(0, max(oe_pct) * 1.30)
ax.text(0.98, 0.98, "close-ended opens:\n"
        + ", ".join(f"{w} {100.0 * c / len(ce_first_words):.0f}%" for w, c in top_ce),
        transform=ax.transAxes, ha="right", va="top", fontsize=ns.fs(6.3), color="#666666",
        linespacing=1.3)
ns.panel_label_below(ax, "g")

# Panel g: PathVQA composition, raw against H&E-filtered.
# Stacked rather than two more threshold bars per series: one bar carries the whole
# distribution, and the pair sits adjacent so the reader sees that filtering moves
# almost nothing - open-ended answers are 85.4% under five words raw, 84.3% filtered.
ax = fig.add_subplot(row2[0, 2])
BAND_COLORS = ["#C8C8C8", "#8C8C8C", ns.OKABE_ITO["vermillion"]]
positions, labels_g = [], []
slot = 0.0
for series in PATHVQA_SERIES:
    for name, source in (("raw", pathvqa_raw), ("H&E", pathvqa)):
        left = 0.0
        for (band, _, _), share, colour in zip(BANDS, composition(source[series]),
                                               BAND_COLORS):
            ax.barh(slot, share, 0.66, left=left, color=colour, edgecolor="white",
                    linewidth=0.3, label=band if slot == 0 else None)
            if share > 11:
                ax.text(left + share / 2, slot, f"{share:.0f}", ha="center",
                        va="center", fontsize=ns.fs(6.8),
                        color="white" if colour != "#C8C8C8" else "#333333")
            left += share
        positions.append(slot)
        labels_g.append(name)
        slot += 1
    slot += 0.5

ax.set_yticks(positions)
ax.set_yticklabels(labels_g, fontsize=ns.fs(7))
ax.invert_yaxis()
ax.set_xlabel("% of items")
ax.set_xlim(0, 100)
ax.set_title("PathVQA: raw vs H&E-filtered", fontsize=7, pad=4)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
ax.tick_params(axis="y", length=0)

# Series names bracket their raw/filtered pair.
series_transform = mtransforms.blended_transform_factory(ax.transAxes, ax.transData)
for index, series in enumerate(PATHVQA_SERIES):
    start, end = positions[index * 2], positions[index * 2 + 1]
    ax.plot([-0.21, -0.235, -0.235, -0.21], [start, start, end, end],
            transform=series_transform, lw=0.6, color="#666666", clip_on=False)
    ax.text(-0.25, (start + end) / 2, series.replace(" ", "\n"),
            transform=series_transform, rotation=90, ha="center", va="center",
            fontsize=ns.fs(6.5), color="#333333", linespacing=1.15)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.235), ncol=3, frameon=False,
          fontsize=ns.fs(7), handletextpad=0.4, columnspacing=0.9, handlelength=1.2)
ns.panel_label_below(ax, "e", dy=-0.44)

paths = ns.save(fig, "figD2_text_characteristics")
print("wrote:", paths)
plt.show()


## Draft caption

**Fig. D2 | Question and answer text characteristics, PathOPEN against PathVQA.**
**a**-**d**, Word-count distributions on a shared axis, each normalised to the percentage
of its own series because the datasets differ in size by two orders of magnitude; the mean
is given in each legend entry. **d** shows PathVQA both raw and after H&E filtering
(dotted and solid), restricted to the 8,992 H&E pairs *before* the word-count floor was
applied - plotting the post-floor set would be circular, since it was selected for length.
**e**, PathVQA word-count composition, raw against H&E-filtered. Filtering barely moves
the distribution: open-ended answers are 85% under five words in the full 32,795-pair
corpus and 84% after filtering, so the length difference from PathOPEN is a property of
PathVQA rather than an artefact of the subset selected for comparison.
**f**, Percentage of PathOPEN items at or above 5 and 10 words, replacing four separate
threshold figures in earlier drafts. MCQ options are the one short category (65% and 48%),
by design: §3.2 mandates 4-6-word options.
**g**, Opening word of PathOPEN's open-ended questions, with the close-ended openings
noted. Questions are predominantly "what"-type, requiring a descriptive diagnosis rather
than a lookup.

*Thresholds are inclusive ($\geq$), matching the dataset-analysis notebooks. MCQ and
close-ended counts are de-duplicated per column, which drops two MCQ questions, three
close-ended questions and ten options that share wording with another item.*
